In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import *

sales = spark.table(
    "agentdb.gold.fact_sales"
)

inventory = spark.table(
    "agentdb.gold.fact_inventory_snapshot"
)

purchase_orders = spark.table(
    "agentdb.gold.fact_purchase_orders"
)

supplier_disruptions = spark.table(
    "agentdb.gold.fact_supplier_disruptions"
)

inventory_health = spark.table(
    "agentdb.gold.snapshot_inventory_health"
)

supplier_risk = spark.table(
    "agentdb.gold.snapshot_supplier_risk"
)

In [0]:
w = Window.partitionBy(
    "product_key",
    "store_key"
).orderBy(
    "calendar_key"
)

daily_velocity = (
    sales

    .withColumn(
        "previous_day_sales_qty",
        lag("sales_qty").over(w)
    )

    .withColumn(
        "sales_velocity",
        (col("sales_qty") -
        col("previous_day_sales_qty")).cast("double")
    )
)

daily_velocity.select(
    "product_key",
    "store_key",
    "calendar_key",
    "sales_qty",
    "previous_day_sales_qty",
    "sales_velocity",
    "created_at"
).write \
    .mode("overwrite") \
    .saveAsTable(
        "agentdb.features.feature_daily_sales_velocity"
    )

In [0]:
w7 = Window.partitionBy(
    "product_key",
    "store_key"
).orderBy(
    "calendar_key"
).rowsBetween(-6, 0)

rolling7 = (

    sales

    .withColumn(
        "rolling_7d_sales",
        sum("sales_qty").over(w7).cast("double")
    )

    .withColumn(
        "rolling_7d_avg",
        avg("sales_qty").over(w7)
    )

    .withColumn(
        "rolling_7d_stddev",
        stddev("sales_qty").over(w7)
    )
)
rolling7.select(
    "product_key",
    "store_key",
    "calendar_key",
    "rolling_7d_sales",
    "rolling_7d_avg",
    "rolling_7d_stddev",
    "created_at"
).write \
    .mode("overwrite") \
    .saveAsTable(
        "agentdb.features.feature_rolling_7d_sales"
    )

In [0]:
w30 = Window.partitionBy(
    "product_key",
    "store_key"
).orderBy(
    "calendar_key"
).rowsBetween(-29, 0)

rolling30 = (
    sales

    .withColumn(
        "rolling_30d_sales",
        sum("sales_qty").over(w30).cast("double")
    )

    .withColumn(
        "rolling_30d_avg",
        avg("sales_qty").over(w30)
    )

    .withColumn(
        "rolling_30d_stddev",
        stddev("sales_qty").over(w30)
    )
)

rolling30.select(
    "product_key",
    "store_key",
    "calendar_key",
    "rolling_30d_sales",
    "rolling_30d_avg",
    "rolling_30d_stddev",
    "created_at"
).write \
    .mode("overwrite") \
    .saveAsTable(
        "agentdb.features.feature_rolling_30d_sales"
    )

In [0]:
days_cover = (
    inventory

    .select(
        "product_key",
        "store_key",
        "calendar_key",
        "inventory_qty",
        "days_of_cover"
    )
)

days_cover.write \
    .mode("overwrite") \
    .saveAsTable(
        "agentdb.features.feature_days_of_cover"
    )

In [0]:
turnover = (

    sales

    .groupBy(
        "product_key",
        "store_key"
    )

    .agg(
        sum("sales_qty")
        .alias("total_sales")
    )

    .join(
        inventory.groupBy(
            "product_key",
            "store_key"
        ).agg(
            avg("inventory_qty")
            .alias(
                "avg_inventory"
            )
        ),
        [
            "product_key",
            "store_key"
        ]
    )

    .withColumn(
        "inventory_turnover_ratio",
        expr("try_divide(total_sales, avg_inventory)")
    )
    .withColumn(
        "calendar_key",
        lit(0).cast("long")
    )
    .withColumn(
        "created_at",
        current_timestamp()
    )
)

turnover.select(
    "product_key",
    "store_key",
    "calendar_key",
    "inventory_turnover_ratio",
    "created_at"
).write \
    .mode("overwrite") \
    .saveAsTable(
        "agentdb.features.feature_inventory_turnover"
    )

In [0]:
stockout_probability = (

    inventory_health

    .withColumn(
        "stockout_probability",

        when(
            col("days_of_cover") < 36,
            0.95
        )
        .when(
            col("days_of_cover") < 37,
            0.70
        )
        .when(
            col("days_of_cover") < 38,
            0.40
        )
        .otherwise(
            0.10
        )
    )
)

stockout_probability.withColumn(
    "forecast_demand",
    col("inventory_qty") / col("days_of_cover") * 7
).withColumn(
    "calendar_key",
    lit(0).cast("long")
).select(
    "product_key",
    "store_key",
    "calendar_key",
    "inventory_qty",
    "forecast_demand",
    "stockout_probability",
    "created_at"
).write \
    .mode("overwrite") \
    .saveAsTable(
        "agentdb.features.feature_stockout_probability"
    )

In [0]:
supplier_risk_feature = (

    supplier_risk

    .select(
        "supplier_key",
        "risk_score"
    )
)

supplier_risk_feature.write \
    .mode("overwrite") \
    .saveAsTable(
        "agentdb.features.feature_supplier_risk_score"
    )

In [0]:
delay_frequency = (

    purchase_orders

    .groupBy(
        "supplier_key"
    )

    .agg(
        count("*").cast("integer")
        .alias("total_pos"),

        sum(
            when(
                col("delay_days") > 0,
                1
            ).otherwise(0)
        ).cast("integer").alias(
            "delayed_pos"
        )
    )

    .withColumn(
        "delay_frequency",
        col("delayed_pos")
        / col("total_pos")
    )
)

delay_frequency.withColumn(
    "created_at",
    current_timestamp()
).select(
    "supplier_key",
    "delayed_pos",
    "total_pos",
    "delay_frequency",
    "created_at"
).write \
    .mode("overwrite") \
    .saveAsTable(
        "agentdb.features.feature_supplier_delay_frequency"
    )

In [0]:
demand_variability = (

    sales

    .groupBy(
        "product_key",
        "store_key"
    )

    .agg(
        avg("sales_qty")
        .alias("avg_daily_sales"),

        stddev("sales_qty")
        .alias("demand_stddev")
    )

    .withColumn(
        "coefficient_of_variation",

        col("demand_stddev")
        /
        col("avg_daily_sales")
    )
    .withColumn(
        "calendar_key",
        lit(0).cast("long")
    )
    .withColumn(
        "created_at",
        current_timestamp()
    )
)

demand_variability.select(
    "product_key",
    "store_key",
    "calendar_key",
    "avg_daily_sales",
    "demand_stddev",
    "coefficient_of_variation",
    "created_at"
).write \
    .mode("overwrite") \
    .saveAsTable(
        "agentdb.features.feature_demand_variability"
    )

In [0]:
from pyspark.sql.functions import col, lag, current_timestamp
from pyspark.sql.window import Window

prices = spark.table(
    "agentdb.silver.price"
)

price_window = Window.partitionBy(
    "product_id",
    "store_id"
).orderBy(
    "wm_yr_wk"
)

price_changes = (

    prices

    .withColumn(
        "previous_price",
        lag("sell_price")
        .over(price_window)
    )

    .withColumn(
        "price_change_pct",

        (
            col("sell_price")
            -
            col("previous_price")
        )
        /
        col("previous_price")
    )
    .withColumn(
        "created_at",
        current_timestamp()
    )
)

price_changes.select(
    col("product_id").alias("product_key"),
    col("store_id").alias("store_key"),
    col("wm_yr_wk").cast("long").alias("calendar_key"),
    col("sell_price").alias("current_price"),
    "previous_price",
    "price_change_pct",
    "created_at"
).write \
    .mode("overwrite") \
    .saveAsTable(
        "agentdb.features.feature_price_change"
    )

In [0]:
tables = [
    "agentdb.features.feature_daily_sales_velocity",
    "agentdb.features.feature_rolling_7d_sales",
    "agentdb.features.feature_rolling_30d_sales",
    "agentdb.features.feature_days_of_cover",
    "agentdb.features.feature_inventory_turnover",
    "agentdb.features.feature_stockout_probability",
    "agentdb.features.feature_supplier_risk_score",
    "agentdb.features.feature_supplier_delay_frequency",
    "agentdb.features.feature_demand_variability",
    "agentdb.features.feature_price_change"
]

from pyspark.sql.functions import lit

counts = [
    spark.table(tbl).select(lit(tbl).alias("table_name"), count("*").alias("row_count"))
    for tbl in tables
]

result = counts[0]
for c in counts[1:]:
    result = result.unionByName(c)

display(result)